# 🔴 Solution: GRPO Loss

In [ ]:
import torch

In [ ]:
# ✅ SOLUTION

def grpo_loss(old_log_probs, new_log_probs, rewards, group_ids, clip_eps: float) -> torch.Tensor:
    advantages = torch.zeros_like(rewards)
    for gid in torch.unique(group_ids):
        mask = group_ids == gid
        group_rewards = rewards[mask]
        advantages[mask] = (group_rewards - group_rewards.mean()) / (group_rewards.std(unbiased=False) + 1e-8)

    ratio = torch.exp(new_log_probs - old_log_probs)
    unclipped = ratio * advantages
    clipped = torch.clamp(ratio, 1.0 - clip_eps, 1.0 + clip_eps) * advantages
    return -torch.minimum(unclipped, clipped).mean()


In [ ]:
# Verify
old_log_probs = torch.log(torch.tensor([0.3, 0.4, 0.2, 0.5]))
new_log_probs = torch.log(torch.tensor([0.33, 0.36, 0.25, 0.45]))
rewards = torch.tensor([1.0, 3.0, 2.0, 6.0])
group_ids = torch.tensor([0, 0, 1, 1])
print(grpo_loss(old_log_probs, new_log_probs, rewards, group_ids, clip_eps=0.2))
